### Plan for Q3 -- to review and revise before any code is written

#### What the question asks for

> What is the most informed action / series of actions in the dataset? For the person that
> made such an action, what was an "economic cost or risk" that they exposed themselves to
> while making this action, or how might this action have ended poorly (beyond simply
> losing more money if they lost the trade)?

Two deliverables, and the second is the one that is not a standard screen:

1. **Name a specific action or series of actions** -- an actual timestamp, market(s), size
   and direction out of this dataset, not a method for finding them.
2. **Say what it cost the person to do it, beyond the money at risk on the trade.** The
   prompt explicitly rules out "they might lose more money", so max-loss arithmetic does
   not answer it. It is about the second-order consequences of having to show your hand to
   the market in order to get the position on.

#### What the data supports

| column | use |
| --- | --- |
| `aggressor_buy_flag` | which side **initiated** the trade. Verified against the book: True -> price is the best ask 94.9% of the time (initiator lifted the offer, bought YES); False -> price is the best bid 95.1% (initiator hit the bid, sold YES). |
| `qty`, `price`, `recv_ts_utc` | size, level, time |
| `native_id` | market, and via `parse_chain_and_strike` the chain and strike |
| `exchange_trade_id` | unique per trade -- usable as a key, but links nothing across trades |
| `player_id`, `player_name` | **100% null** -- no trader identity, no per-trader history, no "watch this account going forward" |

Two things this rules out. There is no way to track an account across trades, which kills
the strongest real-world approach. And `aggressor_buy_flag` does **not** separate makers
from takers -- every trade has one of each by definition -- so we cannot filter for
"aggressive" trades. Aggression has to be measured as sweeping multiple price levels
instead, which is rare: only 44 bursts in the whole dataset cross more than one level.

Every market is an "Over": `outcome_name` takes only the values `Over`,
`Milwaukee Brewers;Over`, `Pittsburgh Pirates;Over`. So buying YES anywhere means "more
runs", and direction aligns across all chains with no sign-flipping.

The tape is lopsided: 1,704 initiated sells against 528 initiated buys, roughly 76/24.

#### The core idea: score the basket, not the market

An informed trader with a view on run scoring does not express it in one market. They
spray across strikes and across correlated chains. Scoring each market against its own
size distribution would therefore miss exactly the pattern we are hunting -- six 500-lot
trades spread over six strikes look unremarkable individually and are one 3,000-lot action
in reality.

So the unit of analysis is a **basket of similar trades in the same direction inside a
short window**, and the reference distribution is built from those basket totals.

#### Defining "similar": one family of 19 markets

The family is **{RFI, F5TOTAL, TOTAL}** -- 19 markets, 2,084 trades, 466,154 contracts.
Everything inside it counts as similar to everything else. RFI is runs in the 1st inning,
F5TOTAL is runs through 5 innings, TOTAL is runs in the whole game: all three are the same
quantity measured over nested windows, and Q1 established the pathwise nesting of RFI
within F5 within TOTAL. A trader who thinks this game will be high-scoring can express that
view in any of the 19 markets, so a burst across them is one action.

The two team-total chains are **excluded from the analysis entirely**, not merely kept
separate:

| chain | strikes | trades | contracts |
| --- | --- | --- | --- |
| TEAMTOTAL-MIL | 7 | 121 | 14,681 |
| TEAMTOTAL-PIT | 7 | **27** | **1,270** |

They are not liquid enough for an informed trader to get size on -- PIT traded 1,270
contracts across 27 trades in six hours -- so an informed trader could not make meaningful
money there even holding a correct view. There is also far too little data to establish
what an outlier even looks like in those chains. Dropping them costs us nothing and avoids
reporting a "finding" resting on 27 observations.

They would also not belong in the family even if they were liquid. MIL runs and PIT runs
are different random variables, and a trader bullish on MIL is not thereby bullish on PIT.

#### Two different windows, and they are easy to confuse

Two time windows appear below. They are unrelated, they point in opposite directions in
time, and they answer different questions, so they are named explicitly throughout rather
than both being called "the window":

| name | what it decides | direction in time | setting |
| --- | --- | --- | --- |
| **grouping window** | are these trades one action by one person? | groups trades happening at the same moment across strikes and chains | **1 second** |
| **payoff window** | did that action actually turn out to be informed? | looks forward from the action to see where the market went | **5 minutes** |

The grouping window collects trades *into* a basket. The payoff window measures what
happened *after* the basket. Getting a large basket from the first tells us someone acted
decisively; only the second tells us whether they were right.

#### Step 1 -- the grouping window: bucket the tape into 1-second intervals

Cut the timeline into consecutive, non-overlapping 1-second buckets, and within each
bucket sum quantity separately for each direction. Each bucket-direction pair becomes one
observation.

Non-overlapping matters. A sliding 1-second look-back anchored on every trade would let one
event produce one observation per print: the 23:32:07 event is 39 prints in a single
second, so it would appear 39 times, filling the leaderboard with copies of itself and
stuffing the reference distribution with 39 near-duplicates of the outlier we are trying to
detect. Consecutive buckets give one event, one row.

Checked against gap-based clustering (break when more than 1 second passes with no
same-direction trade) and the two agree closely -- 307 vs 303 buy observations, 925 vs 851
sell observations, identical maxima -- and the largest event does not straddle a bucket
boundary. Fixed buckets are simpler, so we use those.

Result at 1 second: 1,232 observations, 307 buy-side and 925 sell-side.

**The grouping window is a parameter, not a decision.** The function takes it as an argument and we
run the whole analysis at 1s, 5s, 30s and 5min.

**What the grouping window really encodes is an assumption about the trader's execution technology,
and that assumption can never be verified.** There is no trader identity in this data, so
grouping trades into "one action" is always a guess. If a burst hits several markets at
11:00 and another burst hits at 11:05, we cannot tell whether that is two informed traders
or one trader who is not comfortable with the platform, painstakingly checking size,
direction and strike before sending each order. A 1-second window will catch any informed
trader working through an API or an electronic execution platform. A human clicking through
a UI might take five minutes to do the same thing.

The cost of widening is severe, and it is not symmetric:

| grouping window | buckets | trades per bucket | markets per bucket | largest bucket |
| --- | --- | --- | --- | --- |
| 1s | 1,232 | 1.7 | 1.1 | 47 trades |
| 5s | 1,040 | 2.0 | 1.2 | 51 |
| 30s | 650 | 3.2 | 1.5 | 60 |
| 60s | 471 | 4.4 | 1.7 | 75 |
| 5min | 136 | **15.3** | **3.3** | **277** |

The family trades at 5.8 trades per minute, so a five-minute bucket contains about 29
trades purely by construction -- that is not one person, it is the market. The decisive
check: among five-minute sell buckets holding 5 or more trades (64 of 72), the median
*internal* time span is **253 seconds out of a 300-second window**. The trades fill the
window rather than clustering within it. So a wide bucket does not isolate one slow human,
it swallows five minutes of ordinary flow with the slow human somewhere inside, and we
cannot tell which trades were his.

So the two settings answer different questions, and the write-up should say which one it is
claiming:

- **1 second is the primary answer.** At 1.7 trades and 1.1 markets per bucket, a large
  bucket is a genuine event rather than an accumulation of background flow, and the claim
  "one actor did this" is defensible. This matters because the question presupposes a
  single person -- *"for the person that made such an action"*.
- **The wider windows are a robustness check**, and at those settings the claim weakens
  from "one trader did this" to "there was a sustained episode of concentrated
  same-direction pressure here". That is still a meaningful finding, but it is a different
  and weaker statement, and it should not be presented as identifying an individual.

One further caveat when reading the runs: the reference distribution is rebuilt at each
grouping window, so a score at 1s is not comparable to a score at 30s. **Only the rankings
are comparable.** What we are checking is whether the same action wins at every setting,
which would make the finding robust to a parameter we picked arbitrarily.

#### Step 2 -- score each bucket against the running distribution

For each bucket, compare its total quantity against the distribution of all prior bucket
totals. Using only prior buckets keeps the score honest as a real-time statistic, which is
what Q4a will need.

**Baskets are same-direction; the reference distribution is not.** A basket must be
same-direction trades, since that is what makes it plausibly one actor's action. But the
question being asked of it -- "is this unusually large?" -- is a question about size, not
direction, so the basket is scored against *all* prior baskets regardless of side.

This matters because the buy side is starved. Splitting the distribution by direction
leaves only 307 buy observations across the whole session and just 24 in the first hour,
which is far too few to place a quartile. The two sides are similar enough in shape to
pool:

| | median | p75 | p90 | p99 |
| --- | --- | --- | --- | --- |
| BUY | 30 | 106 | 329 | 10,155 |
| SELL | 43 | 162 | 574 | 5,507 |

Same order of magnitude at every quantile, both heavily fat-tailed. Pooling takes the
reference sample from 307 to 1,232.

The fence is the Tukey rule, `Q3 + 1.5 * IQR`, on that running distribution. Median and
standard deviation are both unusable here: the distribution is extremely fat-tailed (trade
mean 216, standard deviation 1,102, max 27,430), so the large baskets we are hunting
inflate the mean and the standard deviation themselves, raising the bar and hiding the next
one. Quartiles are resistant to exactly that.

The fence is a **screen, not the answer**. Having passed it, buckets are ranked by size
relative to the running median, so the answer is "this basket was N times the typical
basket" rather than a raw contract count.

**Warm-up: a bucket is scoreable once 50 prior observations exist**, and is reported as
unscoreable rather than given a misleading score before that.

The alternative rule -- wait a fixed hour of wall-clock time -- was rejected. With the
distribution split by direction the two rules were both bad: an hour yields only 24 buy
observations, while waiting for 50 buy observations costs 157 minutes, nearly half the
session. Pooling fixes this, since the first hour alone carries 112 pooled buckets, so the
50-observation minimum is met early and costs almost nothing.

A count-based rule is also the one that survives changing the bucket width. An hour-based
rule silently changes meaning when the width changes -- an hour contains sixty times as
many 1-second buckets as 30-second ones -- whereas "50 observations" means the same thing
at every setting.

#### Step 3 -- the payoff window: did the action actually make money?

Large is not the same as informed. Someone can dump 61,000 contracts because they are
rebalancing, or panicking, or simply wrong. The test for "informed" is whether the market
subsequently moved in their favour.

Using `aggressor_buy_flag` for direction, measure how far the market moved the initiator's
way over the payoff window:

    markout = (mid_after - mid_at_action) * (+1 if initiator bought else -1)

Positive markout means the market moved their way, so the action looks informed rather than
merely large.

**The payoff window is 5 minutes.** Fixed in advance, before seeing any results, so that we
cannot end up picking the horizon that flatters whichever action we had already decided we
liked. +1 minute and +30 minutes are reported alongside as context but do not decide the
answer:

- **+1 minute** mostly measures the action's own footprint. Of course the price moved -- they
  just hit fifteen bids. That is impact, not information.
- **+5 minutes** is long enough for the maker to reprice and the footprint to fade, short
  enough that it is still about this action.
- **+30 minutes** should still show genuine information, but it also accumulates half an
  hour of unrelated drift.

If all three agree, the choice does not matter and we say so. If they disagree, 5 minutes
decides.

Note what this step implies: an action can only be confirmed as informed **after the fact**.
That is not a flaw in the method, it is the nature of the problem, and it is exactly the gap
Q4a has to close by finding something observable at the moment of the trade.

Two caveats inherited from Q1 and Q2: the mid must come from a quote that is genuinely fresh
at the measurement point rather than a stale one carried forward, and 66% of trades land
inside book-feed gaps of more than 5 seconds, so some payoff windows will be unmeasurable
and should be reported as missing rather than filled.

#### Step 4 -- name the winner and write the risk section

Leading candidate on the evidence so far: **23:32:07 UTC**, 37 initiated sells totalling
61,241 contracts across RFI and F5TOTAL-4, roughly 8 minutes before first pitch. Next
largest is 41,997 at 21:28:26 on TOTAL-8. Step 3 either confirms it or replaces it -- the
markout decides, not the size.

For whichever basket wins, the economic cost / risk section covers:

1. **Information leakage.** The trade tells everyone watching the tape that someone with
   conviction has taken a side. The edge is worth less the moment it is exercised.
2. **The position cannot be exited.** A 61k basket is a large fraction of everything these
   markets traded in six hours. There is no way out at a sensible price, so the trader is
   locked in to settlement. That converts a view about probability into an all-or-nothing
   outcome at $0 or $1 -- they cannot take the win at 0.80 and go home.
3. **The market maker widens or pulls.** Q1 established that one maker appears to quote the
   whole chain off a single fitted distribution. A large take tells that maker they have
   been adversely selected, so they widen or step away. The trader burns the edge on the
   first clip and cannot repeat it.
4. **The trade telegraphs across the whole ladder.** Because the chain is internally
   consistent -- Q1 found zero arbitrage violations anywhere -- repricing one strike forces
   the maker to reprice every correlated strike. Hitting one market leaks the view into
   eighteen others simultaneously, revealing far more than intended.
5. **Legal and regulatory exposure.** If the edge came from material non-public
   information, the downside is not a losing trade but disgorgement, fines, exchange
   suspension, or prosecution:

   Regulators have made clear that trading event contracts on MNPI (material non-public information) can be prosecuted even though these aren't traditional securities. The CFTC's Enforcement Division put out a Prediction Markets Advisory in February 2026, publicly describing MNPI-based event-contract trading as conduct the CFTC can pursue as "insider trading" under CEA § 6(c)(1) and Rule 180.1. The legal theory used is misappropriation of confidential information, not classic securities insider trading, because event contracts fall under commodities law. 
   Snell & Wilmer

   Actual cases so far
   - The YouTube editor case (Kalshi, 2025) — a trader who was a YouTube channel editor had advanced knowledge of video contents before they posted, and traded on that. Kalshi hit him with a $20,397.58 penalty (disgorgement plus fine) and a 2-year exchange suspension. 
   Commodity Futures Trading Commission
   Political candidate trading on his own race (Kalshi, May 2025) — a political candidate was found trading on his own candidacy, which Kalshi flagged and disciplined.
   - Michele Spagnuolo / Google engineer (May 2026) — the CFTC and DOJ charged a Google employee with using material nonpublic information to trade Polymarket contracts tied to the browser's "Year in Search" lists — first case involving a private-sector company employee trading on internal work knowledge.

The max-loss arithmetic (100 contracts at 0.70 risks $70) stays as a single framing
sentence, since the prompt explicitly excludes it as the answer.

#### Settled parameters

| parameter | value | why |
| --- | --- | --- |
| family | {RFI, F5TOTAL, TOTAL}, 19 markets | same quantity over nested windows; team totals dropped for illiquidity |
| grouping window | 1 second primary, 5s / 30s / 5min as robustness | 1s keeps a bucket to 1.7 trades, so a large bucket is a real event rather than accumulated background flow |
| payoff window | 5 minutes | past the action's own footprint, short of unrelated drift |
| reference distribution | pooled across both directions | buy side alone carries only 24 observations in the first hour |
| outlier fence | Tukey, Q3 + 1.5 x IQR | quartiles resist the fat tail that the trades we are hunting create |
| warm-up | 50 prior observations | means the same thing at every grouping window, unlike an hour of wall clock |

Nothing above is left open. Ready to code.